### Pre Processing

In [2]:
import nibabel as nib
import numpy as np
import cv2
import SimpleITK as sitk
from skimage import exposure
import os

# ---------- config ----------
img_path = r"D:\MI_BRATS\Pre-operative_TCGA_LGG_NIfTI_and_Segmentations\TCGA-CS-4942\TCGA-CS-4942_1997.02.22_t1.nii.gz"
out_path = r"D:\MI_BRATS\Pre-operative_TCGA_LGG_NIfTI_and_Segmentations\TCGA-CS-4942\TCGA-CS-4942_1997.02.22_t1_preproc_clahe.nii.gz"
# If your 3D array has shape (slices, height, width), use SLICE_AXIS = 0
# If it has shape (height, width, slices) (rare for nibabel), use SLICE_AXIS = 2
SLICE_AXIS = 0
# CLAHE params
CLAHE_CLIP = 2.0
CLAHE_TILE = (8, 8)
# ----------------------------

# 1) load using nibabel (keep affine/header for saving)
img_nifti = nib.load(img_path)
affine = img_nifti.affine
header = img_nifti.header
img_data = img_nifti.get_fdata().astype(np.float32)  # float32 for processing

# Optionally reorder so slices are first axis: we want an array of shape (slices, H, W)
if SLICE_AXIS != 0:
    img_data = np.moveaxis(img_data, SLICE_AXIS, 0)

# 2) N4 Bias Field Correction (SimpleITK expects z,y,x numpy ordering -> OK)
sitk_img = sitk.GetImageFromArray(img_data)                       # float32
sitk_img = sitk.Cast(sitk_img, sitk.sitkFloat32)

# Create a crude mask to speed N4 and protect background (optional but recommended)
# Mask = voxels > tiny threshold
mask = sitk.GetImageFromArray((img_data > 0).astype(np.uint8))
mask = sitk.Cast(mask, sitk.sitkUInt8)

corrected_sitk = sitk.N4BiasFieldCorrection(sitk_img, mask)
img_corrected = sitk.GetArrayFromImage(corrected_sitk).astype(np.float32)

# (If we moved axes earlier, img_corrected already matches slices-first shape)

# 3) Z-score normalization — compute mean/std only over non-zero (brain) voxels
brain_mask = img_corrected > 0
if np.any(brain_mask):
    mean_val = img_corrected[brain_mask].mean()
    std_val = img_corrected[brain_mask].std()
else:
    # fallback to global stats if mask absent
    mean_val = img_corrected.mean()
    std_val = img_corrected.std()

if std_val == 0 or np.isnan(std_val):
    std_val = 1.0

img_norm = (img_corrected - mean_val) / std_val
# keep background as zero (or small negative). We'll re-mask later.

# 4) Contrast stretching using percentiles computed on brain voxels
if np.any(brain_mask):
    p2, p98 = np.percentile(img_norm[brain_mask], (2, 98))
else:
    p2, p98 = np.percentile(img_norm, (2, 98))

# Avoid degenerate percentile range
if p98 == p2:
    p2, p98 = img_norm.min(), img_norm.max()
    if p98 == p2:
        p2 -= 1.0
        p98 += 1.0

img_stretched = exposure.rescale_intensity(img_norm, in_range=(p2, p98), out_range=(0.0, 1.0))

# 5) Convert to uint8 (0-255) for CLAHE
img_uint8 = (img_stretched * 255.0).clip(0, 255).astype(np.uint8)

# 6) Apply CLAHE slice-wise (2D). CLAHE.apply requires single-channel uint8 images.
clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)
slices = img_uint8.shape[0]
clahe_slices = []
for i in range(slices):
    img_slice = img_uint8[i]            # shape (H, W)
    # If slice is all zero, CLAHE.apply will still work but has no effect; we can skip if desired
    if np.all(img_slice == 0):
        clahe_slices.append(img_slice)
    else:
        clahe_slices.append(clahe.apply(img_slice))

img_clahe = np.stack(clahe_slices, axis=0).astype(np.uint8)

# 7) Move axes back to original orientation if needed, and save
if SLICE_AXIS != 0:
    img_clahe = np.moveaxis(img_clahe, 0, SLICE_AXIS)

# Save as NIfTI using original affine and header (we use uint8 data)
out_img = nib.Nifti1Image(img_clahe, affine, header=header)
# Optionally set datatype in header
out_img.set_data_dtype(np.uint8)
nib.save(out_img, out_path)

print(f"Saved preprocessed image with CLAHE to: {out_path}")

Saved preprocessed image with CLAHE to: D:\MI_BRATS\Pre-operative_TCGA_LGG_NIfTI_and_Segmentations\TCGA-CS-4942\TCGA-CS-4942_1997.02.22_t1_preproc_clahe.nii.gz


1. Load NIfTI image using Nibabel
2. Extract affine and header information
3. Convert image data to float32
4. Reorder axes so slices come first (if needed)
5. Create SimpleITK image from NumPy array
6. Generate binary mask for foreground voxels
7. Apply N4 bias field correction
8. Convert corrected image back to NumPy array
9. Create brain mask for nonzero voxels
10. Compute mean and standard deviation of brain voxels
11. Apply z-score normalization
12. Compute 2nd and 98th percentiles for contrast stretching
13. Rescale intensity to [0, 1] range
14. Convert image to uint8 (0–255 scale)
15. Initialize CLAHE with clip limit and tile size
16. Apply CLAHE slice-by-slice in 2D
17. Stack processed slices back into 3D array
18. Revert axes to original order (if changed)
19. Create NIfTI image with original affine and header
20. Save preprocessed image to output path
